In [ ]:
import nomenclature
import pyam
import ixmp4

In [ ]:
platform = ixmp4.Platform("scenariocompass-dev")

In [ ]:
variables = platform.iamc.variables.tabulate().name

In [ ]:
[i for i in variables if "Carbon" in i]

In [ ]:
variables = [
    "Emissions|CO2"
    "Carbon Capture|Geological Storage",
]

In [ ]:
df = pyam.read_ixmp4(platform, variable=variables, region="World")

In [ ]:
def cumulative_emissions(x):
    return pyam.timeseries.cumulative(x, 2020, 2100)

In [ ]:
meta_ccs_name = "Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]"

In [ ]:
df.set_meta(
    name="Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]",
    meta= df.timeseries().apply(cumulative_emissions, raw=False, axis=1) / 1000
)

In [ ]:
df.meta.columns

In [ ]:
df.set_meta(name="Reason For Concern|Exceeding Prudent Limit For Geological Carbon Storage|World", meta="ok")

In [ ]:
df.set_meta(
    name="Reason For Concern|Exceeding Prudent Limit For Geological Carbon Storage|World",
    meta="high",
    index=df.meta[df.meta[meta_ccs_name] > 1460].index
)

In [ ]:
df.meta[meta_ccs_name]

In [ ]:
df.set_meta(
    name="Reason For Concern|Exceeding Prudent Limit For Geological Carbon Storage|World",
    meta="medium",
    index=df.meta[(df.meta[meta_ccs_name] < 1460) & (df.meta[meta_ccs_name] > 1290)].index,
)

In [ ]:
missing = df.require_data(variable="Carbon Capture|Geological Storage")

In [ ]:
missing.scenario.unique()

In [ ]:
platform = ixmp4.Platform("scenariocompass-dev")

In [ ]:
for model, scenario in df.index:
    run = platform.runs.get(model=model, scenario=scenario)
    for meta_col in [
        "Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]",
        "Reason For Concern|Exceeding Prudent Limit For Geological Carbon Storage|World"
    ]:
        run.meta[meta_col] = df.meta.loc[(model, scenario), meta_col]
    print(f"Success {model} - {scenario}")
